# Representation Axis and Effective-Rank Analysis for IF vs Math Task Vectors

This notebook focuses on localization features beyond norm/cos conflict:

- Layer-wise effective rank of task-vector matrices
- PCA-style principal subspace alignment
- Linear CKA alignment between task vectors
- Pearson / cosine similarity and sign conflict ratios
- Layer risk localization table for targeted merge interventions


## Metric Design Notes

- **Should task vectors be normalized?**
  - For *alignment* metrics (PCA subspace overlap, Pearson, cosine), normalization is useful and applied implicitly by the metric.
  - For *magnitude* diagnostics (`||Δ||`), raw values are preserved to keep true update intensity.

- **Is Pearson correlation useful?**
  - Yes, as a secondary signal. Pearson captures linear co-variation after centering, while cosine is scale-invariant without centering.
  - Agreement between Pearson and cosine strengthens confidence; disagreement often indicates mean-shift effects.


In [ ]:
from __future__ import annotations

import gc
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

BASE_MODEL_ID = "Qwen/Qwen3-1.7B"
IF_MODEL_PATH = Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface")
MATH_MODEL_PATH = Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface")

# Restrict to core decoder matrices by default for stable representation-axis analysis.
ONLY_CORE_LAYER_WEIGHTS = True
TOPK_SUBSPACE = 8
TAU_RMS_FACTOR = 0.1

ARTIFACT_DIR = Path("merging_analysis/artifacts/representation_axis")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for required_path in [IF_MODEL_PATH, MATH_MODEL_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required checkpoint path does not exist: {required_path}")

sns.set_theme(style="whitegrid", context="talk")
print(f"Artifacts will be written to: {ARTIFACT_DIR}")


In [ ]:
def parse_layer_name(param_name: str) -> str:
    """Map a parameter name to a stable layer identifier."""

    match = re.search(r"model\.layers\.(\d+)\.", param_name)
    if match:
        return f"layer_{int(match.group(1)):02d}"
    if param_name.startswith("model.embed_tokens"):
        return "layer_embed"
    if param_name.startswith("model.norm"):
        return "layer_final_norm"
    if param_name.startswith("lm_head"):
        return "layer_lm_head"
    return "layer_other"


def layer_sort_key(layer_name: str) -> Tuple[int, str]:
    """Sort decoder layers numerically, then special layers."""

    match = re.fullmatch(r"layer_(\d+)", layer_name)
    if match:
        return (0, int(match.group(1)))
    return (1, layer_name)


def load_causal_lm(model_name_or_path: str | Path, dtype: torch.dtype = torch.float16):
    """Load model on CPU for parameter-space representation analysis."""

    return AutoModelForCausalLM.from_pretrained(
        str(model_name_or_path),
        device_map="cpu",
        low_cpu_mem_usage=True,
        torch_dtype=dtype,
        trust_remote_code=True,
    )


def should_use_parameter(param_name: str, param_tensor: torch.Tensor, only_core: bool) -> bool:
    """Filter parameters for representation analysis.

    Args:
        param_name: Parameter name.
        param_tensor: Parameter tensor.
        only_core: If True, keep only decoder block weight matrices.

    Returns:
        Boolean indicating whether the parameter should be analyzed.
    """

    if not torch.is_floating_point(param_tensor):
        return False

    if only_core:
        return (
            ("model.layers." in param_name)
            and param_name.endswith(".weight")
            and (param_tensor.ndim >= 2)
        )

    return param_tensor.ndim >= 1


def tensor_to_matrix(tensor: torch.Tensor) -> torch.Tensor:
    """Convert a tensor into a 2D matrix representation for SVD/PCA-like analysis."""

    if tensor.ndim == 1:
        return tensor.reshape(1, -1)
    return tensor.reshape(tensor.shape[0], -1)


def effective_rank_from_singular_values(singular_values: torch.Tensor, eps: float = 1e-12) -> float:
    """Compute effective rank as `exp(H(p))`, where `p` is normalized singular spectrum."""

    s = torch.clamp(singular_values, min=0.0)
    total = torch.sum(s)
    if total <= eps:
        return 0.0
    p = s / total
    entropy = -torch.sum(p * torch.log(p + eps))
    return float(torch.exp(entropy).item())


def principal_subspace_alignment(matrix_a: torch.Tensor, matrix_b: torch.Tensor, topk: int = 8) -> float:
    """Measure top-k principal subspace overlap via principal angle cosines.

    Returns value in [0, 1], higher is better aligned.
    """

    if matrix_a.numel() == 0 or matrix_b.numel() == 0:
        return 0.0

    _, _, vha = torch.linalg.svd(matrix_a, full_matrices=False)
    _, _, vhb = torch.linalg.svd(matrix_b, full_matrices=False)

    ka = min(topk, vha.shape[0])
    kb = min(topk, vhb.shape[0])
    k = min(ka, kb)
    if k <= 0:
        return 0.0

    basis_a = vha[:k].transpose(0, 1)
    basis_b = vhb[:k].transpose(0, 1)

    sigma = torch.linalg.svdvals(basis_a.transpose(0, 1) @ basis_b)
    sigma = torch.clamp(sigma, 0.0, 1.0)
    return float(torch.mean(sigma).item())


def linear_cka(matrix_a: torch.Tensor, matrix_b: torch.Tensor, eps: float = 1e-12) -> float:
    """Compute linear CKA alignment between two matrices.

    This metric is scale-robust and suitable for comparing representation axes.
    """

    if matrix_a.numel() == 0 or matrix_b.numel() == 0:
        return 0.0

    a = matrix_a - matrix_a.mean(dim=0, keepdim=True)
    b = matrix_b - matrix_b.mean(dim=0, keepdim=True)

    cross = a.transpose(0, 1) @ b
    aa = a.transpose(0, 1) @ a
    bb = b.transpose(0, 1) @ b

    numerator = torch.sum(cross * cross)
    denominator = torch.sqrt(torch.sum(aa * aa) * torch.sum(bb * bb) + eps)
    if denominator <= eps:
        return 0.0
    return float((numerator / denominator).item())


def pearson_correlation(flat_a: torch.Tensor, flat_b: torch.Tensor, eps: float = 1e-12) -> float:
    """Compute Pearson correlation between flattened vectors."""

    a = flat_a - flat_a.mean()
    b = flat_b - flat_b.mean()

    denom = torch.sqrt(torch.sum(a * a) * torch.sum(b * b) + eps)
    if denom <= eps:
        return 0.0
    return float((torch.sum(a * b) / denom).item())


In [ ]:
def analyze_representation_metrics(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
    only_core_layer_weights: bool = True,
    topk_subspace: int = 8,
    tau_rms_factor: float = 0.1,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Compute parameter-level and layer-level representation diagnostics.

    Args:
        base_model: Base model checkpoint.
        if_model: IF model checkpoint.
        math_model: Math model checkpoint.
        only_core_layer_weights: If True, analyze decoder weight matrices only.
        topk_subspace: Number of principal vectors used for subspace alignment.
        tau_rms_factor: Threshold scale for sign conflict diagnostics.

    Returns:
        Tuple of `(parameter_dataframe, layer_dataframe)`.
    """

    if_params = dict(if_model.named_parameters())
    math_params = dict(math_model.named_parameters())

    parameter_rows: List[Dict[str, float]] = []

    with torch.no_grad():
        for name, base_param in tqdm(base_model.named_parameters(), desc="Representation metrics"):
            if name not in if_params or name not in math_params:
                continue
            if if_params[name].shape != base_param.shape or math_params[name].shape != base_param.shape:
                raise ValueError(f"Shape mismatch at parameter: {name}")
            if not should_use_parameter(name, base_param, only_core_layer_weights):
                continue

            layer_name = parse_layer_name(name)

            base_fp32 = base_param.detach().to(torch.float32)
            delta_if = if_params[name].detach().to(torch.float32) - base_fp32
            delta_math = math_params[name].detach().to(torch.float32) - base_fp32

            # Basic norm/alignment statistics.
            norm_if = float(torch.linalg.norm(delta_if).item())
            norm_math = float(torch.linalg.norm(delta_math).item())
            dot = float(torch.sum(delta_if * delta_math).item())
            denom = max(norm_if * norm_math, 1e-12)
            cosine = float(max(min(dot / denom, 1.0), -1.0))
            pearson = pearson_correlation(delta_if.reshape(-1), delta_math.reshape(-1))

            # Thresholded sign conflict (tau from local RMS scale).
            rms_if = norm_if / max(math.sqrt(delta_if.numel()), 1e-12)
            rms_math = norm_math / max(math.sqrt(delta_math.numel()), 1e-12)
            tau = tau_rms_factor * 0.5 * (rms_if + rms_math)
            active_tau = (torch.abs(delta_if) > tau) & (torch.abs(delta_math) > tau)
            sign_diff = (torch.sign(delta_if) * torch.sign(delta_math)) < 0
            conflict_tau = active_tau & sign_diff
            conflict_l_tau_mean = float(conflict_tau.float().mean().item())
            conflict_ratio_active = float(conflict_tau.sum().item() / max(active_tau.sum().item(), 1.0))

            # Matrix-form metrics for effective rank and core-axis alignment.
            matrix_if = tensor_to_matrix(delta_if)
            matrix_math = tensor_to_matrix(delta_math)

            s_if = torch.linalg.svdvals(matrix_if)
            s_math = torch.linalg.svdvals(matrix_math)
            effective_rank_if = effective_rank_from_singular_values(s_if)
            effective_rank_math = effective_rank_from_singular_values(s_math)
            subspace_alignment = principal_subspace_alignment(matrix_if, matrix_math, topk=topk_subspace)
            cka = linear_cka(matrix_if, matrix_math)

            parameter_rows.append(
                {
                    "parameter": name,
                    "layer": layer_name,
                    "numel": int(delta_if.numel()),
                    "norm_if": norm_if,
                    "norm_math": norm_math,
                    "cosine": cosine,
                    "pearson": pearson,
                    "conflict_l_tau_mean": conflict_l_tau_mean,
                    "conflict_ratio_active_tau": conflict_ratio_active,
                    "effective_rank_if": effective_rank_if,
                    "effective_rank_math": effective_rank_math,
                    "subspace_alignment_topk": subspace_alignment,
                    "linear_cka": cka,
                }
            )

    param_df = pd.DataFrame(parameter_rows)
    if param_df.empty:
        raise ValueError("No parameters were selected for representation analysis.")

    def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
        values_np = values.to_numpy(dtype=np.float64)
        weights_np = weights.to_numpy(dtype=np.float64)
        return float(np.sum(values_np * weights_np) / max(np.sum(weights_np), 1e-12))

    layer_rows = []
    for layer_name, group in param_df.groupby("layer", sort=False):
        weights = group["numel"]
        layer_rows.append(
            {
                "layer": layer_name,
                "num_parameters": int(group["numel"].sum()),
                "norm_if_l": float(np.sqrt(np.sum(np.square(group["norm_if"].to_numpy(dtype=np.float64))))),
                "norm_math_l": float(np.sqrt(np.sum(np.square(group["norm_math"].to_numpy(dtype=np.float64))))),
                "cos_l": weighted_mean(group["cosine"], weights),
                "pearson_l": weighted_mean(group["pearson"], weights),
                "effective_rank_if_l": weighted_mean(group["effective_rank_if"], weights),
                "effective_rank_math_l": weighted_mean(group["effective_rank_math"], weights),
                "subspace_alignment_topk_l": weighted_mean(group["subspace_alignment_topk"], weights),
                "linear_cka_l": weighted_mean(group["linear_cka"], weights),
                "conflict_l_tau_mean": weighted_mean(group["conflict_l_tau_mean"], weights),
                "conflict_ratio_active_tau": weighted_mean(group["conflict_ratio_active_tau"], weights),
            }
        )

    layer_df = pd.DataFrame(layer_rows)
    layer_df = layer_df.sort_values("layer", key=lambda col: col.map(layer_sort_key)).reset_index(drop=True)

    # Localization score: high conflict + low alignment + high magnitude -> high risk.
    def zscore(series: pd.Series) -> pd.Series:
        std = float(series.std(ddof=0))
        if std < 1e-12:
            return pd.Series(np.zeros(len(series)), index=series.index)
        return (series - float(series.mean())) / std

    norm_total = layer_df["norm_if_l"] + layer_df["norm_math_l"]
    layer_df["localization_risk_score"] = (
        zscore(norm_total)
        + zscore(layer_df["conflict_l_tau_mean"])
        + zscore(layer_df["conflict_ratio_active_tau"])
        - zscore(layer_df["subspace_alignment_topk_l"])
        - zscore(layer_df["linear_cka_l"])
        - zscore(layer_df["cos_l"])
    )

    denom = (layer_df["norm_if_l"] + layer_df["norm_math_l"]).replace(0.0, 1e-12)
    layer_df["norm_if_share"] = layer_df["norm_if_l"] / denom
    layer_df["norm_math_share"] = layer_df["norm_math_l"] / denom

    return param_df, layer_df


In [ ]:
def visualize_representation_metrics(layer_df: pd.DataFrame, output_dir: Path) -> None:
    """Visualize effective rank, axis alignment, and localization risk."""

    if layer_df.empty:
        raise ValueError("Layer dataframe is empty.")

    plot_df = layer_df.copy()
    plot_df["plot_idx"] = range(len(plot_df))

    fig, axes = plt.subplots(2, 2, figsize=(22, 12), constrained_layout=True)

    # Effective rank trends.
    axes[0, 0].plot(plot_df["plot_idx"], plot_df["effective_rank_if_l"], marker="o", label="IF effective rank")
    axes[0, 0].plot(plot_df["plot_idx"], plot_df["effective_rank_math_l"], marker="o", label="Math effective rank")
    axes[0, 0].set_title("Layer-Wise Effective Rank")
    axes[0, 0].set_xlabel("Layer index")
    axes[0, 0].set_ylabel("Effective rank")
    axes[0, 0].legend()

    # Axis alignment and pairwise correlation signals.
    axes[0, 1].plot(plot_df["plot_idx"], plot_df["subspace_alignment_topk_l"], marker="o", label="Subspace alignment")
    axes[0, 1].plot(plot_df["plot_idx"], plot_df["linear_cka_l"], marker="o", label="Linear CKA")
    axes[0, 1].plot(plot_df["plot_idx"], plot_df["cos_l"], marker="o", label="Cosine")
    axes[0, 1].plot(plot_df["plot_idx"], plot_df["pearson_l"], marker="o", label="Pearson")
    axes[0, 1].set_ylim(-1.05, 1.05)
    axes[0, 1].set_title("Core Axis Alignment Metrics")
    axes[0, 1].set_xlabel("Layer index")
    axes[0, 1].set_ylabel("Alignment score")
    axes[0, 1].legend()

    # Conflict map.
    axes[1, 0].plot(plot_df["plot_idx"], plot_df["conflict_l_tau_mean"], marker="o", label="conflict_l(τ)")
    axes[1, 0].plot(plot_df["plot_idx"], plot_df["conflict_ratio_active_tau"], marker="o", label="conflict ratio | active")
    axes[1, 0].set_ylim(0.0, 1.0)
    axes[1, 0].set_title("Sign Conflict Diagnostics")
    axes[1, 0].set_xlabel("Layer index")
    axes[1, 0].set_ylabel("Ratio")
    axes[1, 0].legend()

    # Localization risk ranking.
    risk_sorted = plot_df.sort_values("localization_risk_score", ascending=False).head(15)
    axes[1, 1].barh(risk_sorted["layer"], risk_sorted["localization_risk_score"])
    axes[1, 1].invert_yaxis()
    axes[1, 1].set_title("Top Risk Layers (Localization)")
    axes[1, 1].set_xlabel("Risk score")
    axes[1, 1].set_ylabel("Layer")

    fig_path = output_dir / "representation_axis_overview.png"
    fig.savefig(fig_path, dpi=180)
    plt.show()
    print(f"Saved figure: {fig_path}")

    # Additional scatter for quick root-cause intuition.
    plt.figure(figsize=(9, 7))
    scatter = plt.scatter(
        plot_df["subspace_alignment_topk_l"],
        plot_df["conflict_l_tau_mean"],
        c=plot_df["norm_if_l"] + plot_df["norm_math_l"],
        cmap="viridis",
        s=90,
    )
    plt.colorbar(scatter, label="Total raw norm")
    plt.xlabel("Subspace alignment (higher is better)")
    plt.ylabel("conflict_l(τ)")
    plt.title("Layer Conflict vs Axis Alignment")
    scatter_path = output_dir / "conflict_vs_alignment_scatter.png"
    plt.savefig(scatter_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved figure: {scatter_path}")


In [ ]:
# --------------------------------------------------------------------------------------
# Execute representation rank/alignment localization analysis
# --------------------------------------------------------------------------------------
base_model = load_causal_lm(BASE_MODEL_ID, dtype=torch.float16)
if_model = load_causal_lm(IF_MODEL_PATH, dtype=torch.float16)
math_model = load_causal_lm(MATH_MODEL_PATH, dtype=torch.float16)

param_df, layer_df = analyze_representation_metrics(
    base_model=base_model,
    if_model=if_model,
    math_model=math_model,
    only_core_layer_weights=ONLY_CORE_LAYER_WEIGHTS,
    topk_subspace=TOPK_SUBSPACE,
    tau_rms_factor=TAU_RMS_FACTOR,
)

param_csv = ARTIFACT_DIR / "parameter_level_representation_metrics.csv"
layer_csv = ARTIFACT_DIR / "layer_level_representation_metrics.csv"
param_df.to_csv(param_csv, index=False)
layer_df.to_csv(layer_csv, index=False)

print(f"Saved parameter metrics: {param_csv}")
print(f"Saved layer metrics: {layer_csv}")

display(layer_df)
visualize_representation_metrics(layer_df=layer_df, output_dir=ARTIFACT_DIR)

# Cleanup for notebook stability.
del base_model
if "if_model" in locals():
    del if_model
if "math_model" in locals():
    del math_model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Print the highest-priority localization targets.
top_layers = layer_df.sort_values("localization_risk_score", ascending=False).head(12)
print("Top localization candidates (high risk):")
display(top_layers)
